In [ ]:
# CODE GENERATOR AGENT


Problem Statement

Role: Strategist at a leading e-commerce company.

Task: Prepare for the upcoming festive season by analyzing competitors' discounts and promotional offers.

Objective: Design a competitive strategy to either match or surpass the competition's offers.

Solution: Develop an Agentic System with the following features:

Dynamic Coding: Ability to adapt to changes in website structure.

Web Scraping: Targeted extraction of discounts and promotional offers from competitor platforms.

Data Summarization: Process and organize extracted information for actionable insights.

Outcome: Enable data-driven decision-making to ensure a competitive edge during the festive season.

In [ ]:
!pip install autogen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.3 MB/s eta 0:00:00


In [ ]:
import autogen
import os
from IPython import get_ipython
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent
from autogen.coding import LocalCommandLineCodeExecutor
from IPython.display import HTML, display

In [ ]:
from getpass import getpass
OPENAI_KEY = getpass("Enter your api key here : ")

Enter your api key here : ··········


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

In [ ]:
# Agent1
user_proxy = UserProxyAgent(
    name="User",
    llm_config=False,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
    human_input_mode="TERMINATE",
    code_execution_config=False
)



In [ ]:
# Agent2 - Developer/Software/PythonDeveloper
code_writer_agent = ConversableAgent(
    name="CodeWriter",
    system_message="""You are a Python developer.
    You use your coding skill to solve problems.
    Once the task is done, returns 'TERMINATE'.""",
    llm_config={"config_list": [{"model": "gpt-4o-mini"}]},
)

In [ ]:
# Agent3 - Reviewer / Quality Checker
code_reviewer_agent = ConversableAgent(
    name="CodeReviewer",
    system_message="""You are an experienced Python software developer
    Review the code given to you based on the requirements.
    Once the task is done, returns 'TERMINATE'.""",
    llm_config={"config_list": [{"model": "gpt-4o-mini"}]},
)

# It's basically a quality checker. "Act as an experienced Python developer and review the code that someone else has written."

In [ ]:
local_executor = LocalCommandLineCodeExecutor(
    timeout=20,
    work_dir='./code files')

In [ ]:
# Agent4
local_executor_agent = ConversableAgent(
    "local_executor_agent",
    llm_config=False,
    code_execution_config={"executor": local_executor},
    human_input_mode="ALWAYS",
)



In [ ]:
messages = ["""To check whether there are any offers or discounts available on a given e-commerce website -
                https://www.flipkart.com/
                Follow these steps,
                1. download the html page of the given URL
                2. we only need text content, so remove any CSS, JavaScript, and Image content
                3. save the remaining html content.
               """ ,
    """The code is given by an LLM. Review the code based on the above requirements and provide it for execution"""]

In [ ]:
chat_results = autogen.initiate_chats(
    [
        {
            "sender": user_proxy,
            "recipient": code_writer_agent,
            "message": messages[0],
            "summary_method": "last_msg",
        },
        {
            "sender": local_executor_agent,
            "recipient": code_reviewer_agent,
            "message": messages[0] + messages[1],
            "summary_method": "last_msg",
        }
    ]
)


********************************************************************************
Starting a new chat....

********************************************************************************
User (to CodeWriter):

To check whether there are any offers or discounts available on a given e-commerce website -
                https://www.flipkart.com/
                Follow these steps,
                1. download the html page of the given URL
                2. we only need text content, so remove any CSS, JavaScript, and Image content
                3. save the remaining html content.
               

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
CodeWriter (to User):

To achieve the task of checking for offers or discounts available on the given e-commerce website, we can follow these steps in Python:

1. Download the HTML page from the specified URL.
2. Use BeautifulSoup to parse the HTML content and remove any CSS, JavaScr

In [ ]:
chat_results = chat_results[1].chat_history
# this will store the last msg as per our summary metthod

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
def fetch_flipkart_content():
    # Step 1: Download the HTML page
    url = "https://www.flipkart.com/"
    try:
        response = requests.get(url)

        # Check if the request was successful
        if response.status_code == 200:
            html_content = response.text

            # Step 2: Remove CSS, JavaScript, and image content
            soup = BeautifulSoup(html_content, 'html.parser')

            # Remove script, style and img elements
            for tag in soup(["script", "style", "img"]):
                tag.decompose()

            # Get text content
            text_content = soup.get_text(separator="\n")

            # Step 3: Save the remaining HTML content
            with open("flipkart_text_content.html", "w", encoding='utf-8') as file:
                file.write(text_content)

            print("HTML content saved successfully.")
        else:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

In [ ]:
# Call the function to execute
fetch_flipkart_content()

HTML content saved successfully.


In [ ]:
chat_results

[ChatResult(chat_id=333717918630810493034445839208466269824, chat_history=[{'content': 'To check whether there are any offers or discounts available on a given e-commerce website -\n                https://www.flipkart.com/\n                Follow these steps,\n                1. download the html page of the given URL\n                2. we only need text content, so remove any CSS, JavaScript, and Image content\n                3. save the remaining html content.\n               ', 'role': 'assistant', 'name': 'User'}, {'content': 'To achieve the task of checking for offers or discounts available on the given e-commerce website, we can follow these steps in Python:\n\n1. Download the HTML page from the specified URL.\n2. Use BeautifulSoup to parse the HTML content and remove any CSS, JavaScript, and image content.\n3. Save the remaining text content to a file.\n\nHere\'s the code to accomplish this:\n\n```python\nimport requests\nfrom bs4 import BeautifulSoup\n\n# Step 1: Download th

# GROUP CHAT

Problem Statement
New Project Focus: A mobile gaming company is developing a game for PS5, requiring new expertise.

HR Role: The HR team is responsible for determining hiring requirements by collaborating with relevant departments.

Agentic System: Implement a group chat system where HR initiates discussions with a Group Chat Manager to gather hiring needs from team members.

In [ ]:
# configuration for LLM
config_list = {
    "config_list": [{"model": "gpt-4o-mini"}]
}

In [ ]:
# Agent1- initiate conversation
hr_agent = ConversableAgent(
    "hr_agent",
    system_message="""You are the head of HR in a company. The company is initiating the development of
                    a new B2C tech product and aims to assemble a dedicated team for this project.
                    Gather the hiring requirements from different departments as needed.""",
    llm_config=config_list,
    human_input_mode="NEVER",
)

In [ ]:
# Agent2
product_agent = ConversableAgent(
    "product_agent",
    system_message="""You are the Head of Product at a company developing a new B2C tech product.
        Your goal is to communicate with the HR department to outline the hiring requirements for assembling a
        dedicated Product management team for this project. Clearly and concisely list the key roles, qualifications, and
        skills, and budget needed for this initiative.""",
    llm_config=config_list,
    human_input_mode="NEVER",
)

In [ ]:
# Agent3
engineering_agent = ConversableAgent(
    "engineering_agent",
    system_message="""You are the Head of Engineering at a company developing a new B2C tech product.
        Your goal is to communicate with the HR department to outline the hiring requirements for assembling a
        dedicated engineering team for this project. Clearly and concisely list the key roles, qualifications, and
        skills, and budget needed for this initiative.""",
    llm_config=config_list,
    human_input_mode="NEVER",
)

In [ ]:
# Agent4
marketing_agent = ConversableAgent(
    "marketing_agent",
    system_message="""You are the Head of Marketing at a company developing a new B2C tech product.
        Your goal is to communicate with the HR department to outline the hiring requirements for assembling a
        dedicated marketing team for this project. Clearly and concisely list the key roles, qualifications, and
        skills, and budget needed for this initiative.""",
    llm_config=config_list,
    human_input_mode="NEVER",
)

In [ ]:
# Agent5
customer_support_agent = ConversableAgent(
    "customer_support_agent",
    system_message="""You are the Head of Customer support at a company developing a new B2C tech product.
        Your goal is to communicate with the HR department to outline the hiring requirements for assembling a
        dedicated customer support team for this project. Clearly and concisely list the key roles, qualifications, and
        skills, and budget needed for this initiative.""",
    llm_config=config_list,
    human_input_mode="NEVER",
)

In [ ]:
from autogen import ConversableAgent, GroupChat, GroupChatManager

In [ ]:
from IPython.display import Markdown, display

In [ ]:
group_chat = GroupChat(
    agents=[customer_support_agent, product_agent, engineering_agent, marketing_agent],
    messages=[],
    max_round= 10,
    speaker_selection_method='auto',
    select_speaker_auto_llm_config=config_list
)

In [ ]:
group_chat_manager = GroupChatManager(
    groupchat=group_chat,
)

In [ ]:
chat_result = hr_agent.initiate_chat(
    group_chat_manager,
    message="A gaming company focussed on mobiles games is launching a new game designed for PS5. Provide hiring requirements for this new product",
    summary_method="last_msg",
)

hr_agent (to chat_manager):

A gaming company focussed on mobiles games is launching a new game designed for PS5. Provide hiring requirements for this new product

--------------------------------------------------------------------------------

Next speaker: product_agent

product_agent (to chat_manager):

Subject: Hiring Requirements for Product Management Team – New PS5 Game

Dear HR Team,

As we embark on the development of our new game designed for PS5, it is crucial to assemble a dedicated Product Management team. Below are the key roles, qualifications, skills, and budgetary considerations needed for this initiative.

### Key Roles:

1. **Product Manager**
   - **Responsibilities:** Define product vision, roadmap, and growth opportunities. Collaborate with development, marketing, and design teams.
   - **Qualifications:** Bachelor's degree in Business, Marketing, or related field. 5+ years of experience in product management, ideally within gaming.
   - **Skills:** Strong analyt

In [ ]:
agent_name_to_find = "customer_support_agent"

# Iterate through the dictionary to find the last conversation for the specified agent
for agent, interactions in group_chat_manager.chat_messages.items():
    # Filter interactions by agent name
    filtered_interactions = [i for i in interactions if i.get("name") == agent_name_to_find]
    if filtered_interactions:
        # Print the last conversation's content
        print(filtered_interactions[-1]["content"])
        break
else:
    print("Agent not found or no interactions available.")


Subject: Hiring Requirements for Customer Support Team – New B2C Tech Product

Dear HR Team,

As we prepare for the launch of our new B2C tech product, it is critical to establish a dedicated customer support team to ensure an outstanding experience for our users. Below are the key roles needed, along with qualifications, skills, and budget estimates for this initiative.

### Key Roles:

1. **Customer Support Manager**
   - **Responsibilities:** Oversee the customer support team, develop support policies, and ensure high-quality service delivery.
   - **Qualifications:** Bachelor’s degree in Business, Communications, or related field. 5+ years of experience in customer support roles, with management experience preferred.
   - **Skills:** Strong leadership and interpersonal skills, excellent problem-solving abilities, and proficiency in customer support software (e.g., Zendesk, Freshdesk).

2. **Customer Support Representative**
   - **Responsibilities:** Respond to customer inquiries, 